Assignment 3

**Course:** ADSP 32021

**Student:** Mpiira Tabuti
**CNetID:** mpiira

**Due:** 2026-11-09

<style>.highlight .gp { color: #2e7d32; font-weight: bold; }</style>

### 0. Dependencies and constants

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pathlib import Path

import mlflow
from autogluon.tabular import TabularPredictor
import h2o
from h2o.automl import H2OAutoML
from sklearn.model_selection import train_test_split

RAW_CSV = "athletes.csv"
SEED = 42
TEST_SIZE = 0.2
LIFTS = ["deadlift", "candj", "snatch", "backsq"]
TARGET = "total_lift"
SURVEY = ["eat", "background", "experience", "schedule", "howlong"]
ENTITY = "athlete_id"
NON_FEATURES = LIFTS + [TARGET, ENTITY]   # target, its parts and the ID are never features

TOP_N = 3                                 # "only the top features" runs
TIME_LIMIT = 300                          # seconds per AutoML run
METRIC = "root_mean_squared_error"        # AutoGluon ranking metric (H2O uses RMSE too)
AG_PATH = Path.home() / "automl_runs" / "hw3"   # AutoGluon model files, outside OneDrive
HW2_EXPERIMENT = "athletes_feature_store" # Part A runs in the copied mlflow.db
EXPERIMENT = "athletes_automl"            # MLflow experiment for HW3

## Part B

### B.1 Use athletes.csv

In [4]:
# ---------------- Course-provided cleaning (same as HW2) ----------------
raw = pd.read_csv(RAW_CSV, low_memory=False)

# Remove rows missing key fields, then irrelevant columns
# FIX: duplicate names removed from the dropna list
df = raw.dropna(subset=["region", "age", "weight", "height", "howlong",
                        "gender", "eat", "train", "background", "experience",
                        "schedule", "deadlift", "candj", "snatch", "backsq"])
# FIX: athlete_id kept so the train/test split matches HW2
df = df.drop(columns=["affiliate", "team", "name", "fran", "helen", "grace",
                        "filthy50", "fgonebad", "run400", "run5k", "pullups",
                        "train"])

# Remove outliers
df = df[df["weight"] < 1500]
df = df[df["gender"] != "--"]
df = df[df["age"] >= 18]
df = df[(df["height"] < 96) & (df["height"] > 48)]

# FIX: deadlift cap applied per gender
male_ok = (df["gender"] == "Male") & (df["deadlift"] <= 1105)
female_ok = (df["gender"] == "Female") & (df["deadlift"] <= 636)
df = df[(df["deadlift"] > 0) & (male_ok | female_ok)]
df = df[(df["candj"] > 0) & (df["candj"] <= 395)]
df = df[(df["snatch"] > 0) & (df["snatch"] <= 496)]
df = df[(df["backsq"] > 0) & (df["backsq"] <= 1069)]

# Clean survey data
df = df.replace({"Decline to answer|": np.nan})
df = df.dropna(subset=SURVEY)

# ---------------- Additional sanity filters (beyond the course outliers) ----------------
df = df[df["weight"] >= 80]                       # adults under 80 lb: entry errors
df = df[(df[LIFTS] >= 20).all(axis=1)]            # an empty training bar is 35-45 lb
bmi = 703 * df["weight"] / df["height"] ** 2
df = df[bmi.between(15, 50)]                      # e.g. 5'9" at 474 lb or 100 lb

df[TARGET] = df[LIFTS].sum(axis=1)
df[ENTITY] = df[ENTITY].astype("int64")

# ---------------- Survey encoding (same as HW2 feature version v2) ----------------
# Survey cells are multi-select ("a|b|"); ordered scales keep the HIGHEST level ticked
TENURE = {"Less than 6": 0, "6-12": 1, "1-2": 2, "2-4": 3, "4+": 4}
SPORT = {"no athletic": 0, "recreational": 1, "youth or high school": 2,
            "college": 3, "professional": 4}
DIET = {"convenient": 0, "cheat meals": 1, "quality foods": 1,
        "Paleo": 2, "weigh and measure": 2}
DOUBLES = {"only do 1 workout": 0, "1x a week": 1, "2x a week": 2, "3+ times": 3}
FLAGS = {  # new column: (source column, keyword)
    "trains_others": ("experience", "train other people"),
    "level1_cert": ("experience", "Level 1"),
    "specialty_course": ("experience", "specialty courses"),
    "coached_start": ("experience", "with a coach"),
    "life_changing": ("experience", "life changing"),
    "rests_4plus": ("schedule", "4 or more days"),
    "plans_rest": ("schedule", "strictly schedule"),
}

def answers(cell):
    """Split a pipe-delimited cell into its answers, dropping declines."""
    return [a for a in cell.split("|") if a and "Decline" not in a]

def max_level(cell, levels):
    """Highest level whose keyword appears in the cell (NaN if none)."""
    hits = [lvl for kw, lvl in levels.items() for a in answers(cell) if kw in a]
    return max(hits) if hits else np.nan

def has(cell, keyword):
    """1 if any answer in the cell contains the keyword, else 0."""
    return int(any(keyword in a for a in answers(cell)))

df["bmi"] = 703 * df["weight"] / df["height"] ** 2
df["tenure"] = df["howlong"].apply(max_level, levels=TENURE)
df["sport_level"] = df["background"].apply(max_level, levels=SPORT)
df["diet"] = df["eat"].apply(max_level, levels=DIET)
df["doubles"] = df["schedule"].apply(max_level, levels=DOUBLES).fillna(0)
for new_col, (src, kw) in FLAGS.items():
    df[new_col] = df[src].apply(has, keyword=kw)

# ---------------- Feature table and the same split as HW2 ----------------
# Sorted by athlete_id like HW2's Feast output, so SEED picks the same test athletes
data = df.drop(columns=LIFTS + SURVEY).sort_values(ENTITY).reset_index(drop=True)
train, test = train_test_split(data, test_size=TEST_SIZE, random_state=SEED)
FEATURES = [c for c in data.columns if c not in NON_FEATURES]

print(f"raw {len(raw):,} rows -> clean {len(data):,} rows | {len(FEATURES)} features | "
        f"train {len(train):,} / test {len(test):,}")
print("features:", FEATURES)
data.head()


raw 423,006 rows -> clean 29,859 rows | 17 features | train 23,887 / test 5,972
features: ['region', 'gender', 'age', 'height', 'weight', 'bmi', 'tenure', 'sport_level', 'diet', 'doubles', 'trains_others', 'level1_cert', 'specialty_course', 'coached_start', 'life_changing', 'rests_4plus', 'plans_rest']


,athlete_id,region,gender,age,height,weight,total_lift,bmi,tenure,sport_level,diet,doubles,trains_others,level1_cert,specialty_course,coached_start,life_changing,rests_4plus,plans_rest
0,84,Northern California,Male,25.0,68.0,160.0,1190.0,24.325260,4,2,1,0.0,0,0,0,1,1,0,0
1,86,Northern California,Male,32.0,77.0,225.0,1225.0,26.678192,4,3,2,0.0,1,1,1,0,1,1,0
2,88,Northern California,Male,30.0,72.0,200.0,1543.0,27.121914,4,3,2,2.0,1,1,1,0,1,1,1
3,96,Northern California,Female,30.0,69.0,125.0,523.0,18.457257,3,0,2,0.0,0,1,1,1,0,0,0
4,1618,South Central,Male,31.0,68.0,174.0,1120.0,26.453720,2,2,1,2.0,1,0,0,0,1,1,1


### B.2. Use AutoML to find the best model 

In [ ]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")   # same database as HW2 (Part A runs are in it)
mlflow.set_experiment(EXPERIMENT)

def run_autogluon(features, name):
    """AutoGluon AutoML on the given features; leaderboard and best model logged to MLflow."""
    with mlflow.start_run(run_name=f"autogluon_{name}"):
        predictor = TabularPredictor(label=TARGET, eval_metric=METRIC,
                                        path=str(AG_PATH / name)).fit(
            train[features + [TARGET]], time_limit=TIME_LIMIT, presets="medium_quality")

        # Validation scores come from AutoGluon's own holdout inside train;
        # test scores use the same held-out athletes as HW2
        board = predictor.leaderboard(test[features + [TARGET]], extra_metrics=["r2"])
        board = board.assign(rmse_val=-board["score_val"], rmse_test=-board["score_test"])

        best = board.iloc[0]
        mlflow.set_tags({"tool": "autogluon", "best_model": predictor.model_best})
        mlflow.log_params({"n_features": len(features), "time_limit": TIME_LIMIT,
                            "presets": "medium_quality", "n_models": len(board)})
        mlflow.log_metrics({"best_rmse_val": best["rmse_val"], "best_rmse_test": best["rmse_test"],
                            "best_r2_test": best["r2"], "best_fit_seconds": best["fit_time"]})
        mlflow.log_table(board, artifact_file="leaderboard.json")
    return predictor, board


ag_all, board_all = run_autogluon(FEATURES, "all_features")

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.1
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26300
CPU Count:          12
Pytorch Version:    2.14.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       16.80 GB / 31.67 GB (53.1%)
Disk Space Avail:   172.32 GB / 474.72 GB (36.3%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "C:\Users\mpiir\automl_runs\hw3\all_features"
Train Data Rows:    23887
Train Data Columns: 17
Label Column:       total_lift
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and many unique label-values observed).
	Label info (max, min, mean, stddev): (2135.0, 80.0, 1019.36158, 277.2489)
	If 'regression' is not the correct problem_type, please manually spec

In [10]:
print("best model:", ag_all.model_best)
display(board_all[["model", "rmse_val", "rmse_test", "r2", "fit_time", "pred_time_val"]].round(3))

best model: WeightedEnsemble_L2


,model,rmse_val,rmse_test,r2,fit_time,pred_time_val
0,WeightedEnsemble_L2,134.139,134.065,0.763,137.367,0.289
1,CatBoost,134.691,134.485,0.761,48.785,0.009
2,NeuralNetTorch,135.078,135.051,0.759,73.266,0.033
3,LightGBMXT,135.767,135.206,0.759,1.735,0.029
4,LightGBM,135.560,135.587,0.757,1.328,0.024
5,LightGBMLarge,136.350,136.166,0.755,2.740,0.016
6,XGBoost,135.687,136.537,0.754,0.816,0.019
7,ExtraTreesMSE,139.548,139.279,0.744,6.990,0.219
8,RandomForestMSE,139.902,140.207,0.740,11.412,0.174


### B.3 Data insights from the AutoML run

By validation RMSE, AutoGluon ranks WeightedEnsemble_L2 first (134.1 lb), CatBoost second (134.7 lb) and NeuralNetTorch third (135.1 lb). The LightGBM and XGBoost models come next at about 135.6 to 136.4 lb, and the two bagged tree models, ExtraTreesMSE and RandomForestMSE, come last at about 139.5 to 139.9 lb. This ranking seems to say a few things about the athletes data. First, CatBoost being the best single model fits a dataset with a categorical column that has many levels: CatBoost encodes categories such as `region` internally, while the other models receive them in simpler forms. Second, the neural network almost ties the boosted trees, and the boosted models beat random forests and extra trees. A likely reason is that `total_lift` changes fairly smoothly and steadily with a few strong inputs such as `weight`, `height` and `gender`. Boosting and neural networks can follow that kind of gradual trend closely, while forests average many independent trees and tend to predict in coarser steps. Third, the ensemble improves on CatBoost by only about 0.5 lb, so the models seem to make very similar mistakes on the same athletes. Combining different model types cannot fix errors that all of them share. Together with every model landing in a narrow 134 to 140 lb range, this suggests the remaining error comes mostly from what the features cannot capture, for example training history or technique, rather than from the choice of algorithm.

### B.4 Top 5 features

In [ ]:
# Permutation importance on AutoGluon's internal validation rows (processed features,
# same 17 names as the raw columns); the test set is kept out because the top
# features are reused for the B.5 runs
importance = ag_all.feature_importance(feature_stage="transformed", silent=True)
TOP_FEATURES = importance.index[:TOP_N].tolist()

print("top 3 features (used in B.5):", TOP_FEATURES)
display(importance.head(5).round(3))

top 3 features (used in B.5): ['gender', 'bmi', 'age']


,importance,stddev,p_value,n,p99_high,p99_low
gender,102.375,3.265,0.0,5,109.098,95.652
bmi,24.789,2.068,0.0,5,29.048,20.531
age,23.226,0.773,0.0,5,24.819,21.634
weight,16.226,1.162,0.0,5,18.618,13.833
tenure,9.328,0.250,0.0,5,9.843,8.814


### B.5 Top 3 models per validation score

In [11]:
ag_top, board_top = run_autogluon(TOP_FEATURES, f"top{TOP_N}_features")

# leaderboard() sorts by test score when given test data, so re-sort by validation score
for label, board in [("all features", board_all), (f"top {TOP_N} features", board_top)]:
    print(f"\nTop 3 models by validation RMSE, {label}:")
    display(board.sort_values("rmse_val")[["model", "rmse_val", "rmse_test", "r2"]].head(3).round(3))

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.1
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26300
CPU Count:          12
Pytorch Version:    2.14.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       16.35 GB / 31.67 GB (51.6%)
Disk Space Avail:   172.27 GB / 474.72 GB (36.3%)
Presets specified: ['medium_quality']


Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "C:\Users\mpiir\automl_runs\hw3\top3_features"
Train Data Rows:    23887
Train Data Columns: 3
Label Column:       total_lift
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and many unique label-values observed).
	Label info (max, min, mean, stddev): (2135.0, 80.0, 1019.36158, 277.2489)
	If 'regression' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    16703.76 MB
	Train Data (Original)  Memory Usage: 1.58 MB (0.0% of available memory)
	Inferring data type of each feature b


Top 3 models by validation RMSE, all features:


,model,rmse_val,rmse_test,r2
0,WeightedEnsemble_L2,134.139,134.065,0.763
1,CatBoost,134.691,134.485,0.761
2,NeuralNetTorch,135.078,135.051,0.759



Top 3 models by validation RMSE, top 3 features:


,model,rmse_val,rmse_test,r2
1,WeightedEnsemble_L2,159.021,161.953,0.654
2,NeuralNetTorch,159.234,162.034,0.653
3,LightGBMXT,159.352,162.153,0.653


### B.6 Top 3 models per speed

In [12]:
# Training speed = fit_time (s); prediction speed = pred_time_test (s for 5,972 test rows).
# Ensemble times include all the base models it combines.
for label, board in [("all features", board_all), (f"top {TOP_N} features", board_top)]:
    for col in ["fit_time", "pred_time_test"]:
        print(f"\nTop 3 fastest models by {col}, {label}:")
        display(board.sort_values(col)[["model", "fit_time", "pred_time_test", "rmse_val"]].head(3).round(3))


Top 3 fastest models by fit_time, all features:


,model,fit_time,pred_time_test,rmse_val
6,XGBoost,0.816,0.184,135.687
4,LightGBM,1.328,0.074,135.560
3,LightGBMXT,1.735,0.127,135.767



Top 3 fastest models by pred_time_test, all features:


,model,fit_time,pred_time_test,rmse_val
5,LightGBMLarge,2.740,0.045,136.350
2,NeuralNetTorch,73.266,0.067,135.078
4,LightGBM,1.328,0.074,135.560



Top 3 fastest models by fit_time, top 3 features:


,model,fit_time,pred_time_test,rmse_val
5,XGBoost,0.373,0.190,159.830
4,LightGBM,0.964,0.103,159.772
3,LightGBMXT,1.015,0.078,159.352



Top 3 fastest models by pred_time_test, top 3 features:


,model,fit_time,pred_time_test,rmse_val
6,LightGBMLarge,1.752,0.049,160.955
2,NeuralNetTorch,67.177,0.059,159.234
0,CatBoost,1.823,0.066,159.523


| Rank | All features, training time | All features, prediction time (5,972 rows) | Top 3 features, training time | Top 3 features, prediction time |
|---|---|---|---|---|
| 1 | XGBoost (0.82 s) | LightGBMLarge (0.045 s) | XGBoost (0.37 s) | LightGBMLarge (0.049 s) |
| 2 | LightGBM (1.33 s) | NeuralNetTorch (0.067 s) | LightGBM (0.96 s) | NeuralNetTorch (0.059 s) |
| 3 | LightGBMXT (1.74 s) | LightGBM (0.074 s) | LightGBMXT (1.02 s) | CatBoost (0.066 s) |

Speed is measured in two ways: training time (`fit_time`) and the time to predict the 5,972 test athletes (`pred_time_test`). For training, the gradient boosting libraries XGBoost and LightGBM are the fastest in both runs, and they all train in under 2 seconds. With only 3 features, their training times drop further, for example XGBoost from 0.82 s to 0.37 s, since each tree has fewer columns to search for splits. The biggest change is CatBoost, which went from 48.8 s with all features to 1.8 s with the top 3. A likely reason is that `region`, which has many categories, is no longer in the data. For prediction, all models are very fast (under 0.3 s for almost 6,000 athletes), and the ranking is close enough that it could change between runs. NeuralNetTorch is one of the fastest models at prediction even though it is one of the slowest to train (about 67 to 73 s).

### B.7 Comparison with the Part A model

Part A only recorded train and test scores, not a validation score, so the fairest comparison is test RMSE. Both homeworks use the same 5,972 test athletes, because HW3 uses the same cleaning, the same sort order and the same seed for the split. The best Part A model was the deep random forest on the v2 features (the same 17 features used here). It reached a test RMSE of 139.6 lb (R² 0.743) and took 10.7 s to train. AutoGluon's best model, WeightedEnsemble_L2, reached 134.1 lb (R² 0.763), an improvement of only about 5.5 lb. Even AutoGluon's own RandomForestMSE (140.2 lb) scored almost the same as the Part A forest, which suggests the Part A model was already close to what this feature set allows. On speed, the picture depends on the model. The ensemble took about 137 s to train, because its time includes all the base models it combines, so it is about 13x slower than the Part A forest. CatBoost (48.8 s, 134.5 lb) and NeuralNetTorch (73.3 s, 135.1 lb) are also much slower. However, LightGBM (1.3 s, 135.6 lb) and XGBoost (0.8 s, 136.5 lb) trained faster than even the shallow Part A forest (2.1 s), and they still scored about 3 to 4 lb better than the deep forest. So for this dataset, a single boosted tree model found by AutoML seems to give the best balance of score and speed. The full ensemble buys a small extra gain at a much higher training cost.

### B.8 No-code, low-code or full-code?

AutoGluon is a low-code AutoML platform. It has no graphical interface, so it still has to be run by writing Python. However, the whole search takes only a few lines: `TabularPredictor(label=...).fit(train)` followed by `leaderboard()`. AutoGluon then does the preprocessing (type detection, boolean and category encoding), the validation split, the training of 9 different model types and the weighted ensemble by itself. It is not full-code, because none of the models, hyperparameters or preprocessing steps had to be written by hand, unlike the scikit-learn pipeline built in Part A.

### B.9. Repeat 2 - 5 using H2O AutoML

### B.9.2 Use H2O AutoML to find the best model

In [13]:
h2o.init(nthreads=-1, max_mem_size="8G")
h2o.no_progress()
train_h2o = h2o.H2OFrame(train)   # text columns (region, gender) become H2O categoricals
test_h2o = h2o.H2OFrame(test)

def run_h2o(features, name):
    """H2O AutoML on the given features; leaderboard and best model logged to MLflow."""
    with mlflow.start_run(run_name=f"h2o_{name}"):
        aml = H2OAutoML(max_runtime_secs=TIME_LIMIT, seed=SEED, sort_metric="RMSE")
        aml.train(x=features, y=TARGET, training_frame=train_h2o)

        # Validation scores are H2O's 5-fold cross-validation on train;
        # test scores use the same held-out athletes as HW2 and AutoGluon
        board = h2o.automl.get_leaderboard(aml, extra_columns="ALL").as_data_frame()
        board = board.rename(columns={"rmse": "rmse_val"})
        perf = {m: h2o.get_model(m).model_performance(test_h2o) for m in board["model_id"]}
        board["rmse_test"] = board["model_id"].map(lambda m: perf[m].rmse())
        board["r2"] = board["model_id"].map(lambda m: perf[m].r2())
        board["fit_time"] = board["training_time_ms"] / 1000

        best = board.iloc[0]
        mlflow.set_tags({"tool": "h2o", "best_model": aml.leader.model_id})
        mlflow.log_params({"n_features": len(features), "max_runtime_secs": TIME_LIMIT,
                            "nfolds": 5, "n_models": len(board)})
        mlflow.log_metrics({"best_rmse_val": best["rmse_val"], "best_rmse_test": best["rmse_test"],
                            "best_r2_test": best["r2"], "best_fit_seconds": best["fit_time"]})
        mlflow.log_table(board, artifact_file="leaderboard.json")
    return aml, board

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; Java HotSpot(TM) 64-Bit Server VM (build 21.0.2+13-LTS-58, mixed mode, sharing)
  Starting server from C:\Users\mpiir\AppData\Local\Programs\Python\Python312\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\mpiir\AppData\Local\Temp\tmpnyrtrfxe
  JVM stdout: C:\Users\mpiir\AppData\Local\Temp\tmpnyrtrfxe\h2o_mpiir_started_from_python.out
  JVM stderr: C:\Users\mpiir\AppData\Local\Temp\tmpnyrtrfxe\h2o_mpiir_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,03 secs
H2O_cluster_timezone:,America/New_York
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.12
H2O_cluster_version_age:,2 months and 27 days
H2O_cluster_name:,H2O_from_python_mpiir_oq2gwv
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,7.980 Gb
H2O_cluster_total_cores:,12
H2O_cluster_allowed_cores:,12
H2O_cluster_status:,"locked, healthy"



+------------------------------------------------------------------+
| You are running the community edition of H2O-3 OSS.              |
|                                                                  |
| For commercial use, H2O-3 Secure is now recommended.             |
| This includes production support, CVE fixes, multi-node scaling, |
| model artifact extraction, and more.                             |
| See h2o.ai/h2o-3/oss-vs-secure for additional details.           |
| Contact enterprise@h2o.ai to upgrade.                            |
+------------------------------------------------------------------+


In [14]:
h2o_all, h2o_board_all = run_h2o(FEATURES, "all_features")

print("best model:", h2o_all.leader.model_id)
display(h2o_board_all[["model_id", "algo", "rmse_val", "rmse_test", "r2", "fit_time",
                        "predict_time_per_row_ms"]].round(3))


17:12:16.992: AutoML: XGBoost is not available; skipping it.



c:\Users\mpiir\AppData\Local\Programs\Python\Python312\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


best model: StackedEnsemble_AllModels_3_AutoML_1_20261107_171216


,model_id,algo,rmse_val,rmse_test,r2,fit_time,predict_time_per_row_ms
0,StackedEnsemble_AllModels_3_AutoML_1_20261107_...,StackedEnsemble,138.526,134.594,0.761,0.667,0.028
1,StackedEnsemble_BestOfFamily_4_AutoML_1_202611...,StackedEnsemble,138.567,134.575,0.761,0.305,0.013
2,StackedEnsemble_BestOfFamily_3_AutoML_1_202611...,StackedEnsemble,138.707,135.196,0.759,0.475,0.012
3,StackedEnsemble_AllModels_2_AutoML_1_20261107_...,StackedEnsemble,138.726,135.199,0.759,0.402,0.016
4,StackedEnsemble_AllModels_1_AutoML_1_20261107_...,StackedEnsemble,138.881,135.457,0.758,0.484,0.019
5,GBM_grid_1_AutoML_1_20261107_171216_model_19,GBM,138.942,134.897,0.760,0.544,0.010
6,GBM_grid_1_AutoML_1_20261107_171216_model_6,GBM,138.947,135.208,0.759,0.659,0.013
7,StackedEnsemble_BestOfFamily_1_AutoML_1_202611...,StackedEnsemble,138.949,135.536,0.757,0.617,0.013
8,StackedEnsemble_BestOfFamily_2_AutoML_1_202611...,StackedEnsemble,138.969,135.515,0.757,0.643,0.032
9,GBM_grid_1_AutoML_1_20261107_171216_model_14,GBM,139.053,135.288,0.758,0.645,0.011


### B.9.3 Data insights from the H2O AutoML run

In 300 seconds, H2O trained 49 models. Most of them are gradient boosting machines (GBM), many coming from a random grid search over GBM settings. There are also several deep learning models, a random forest (DRF), an extremely randomized forest (XRT), one linear model (GLM) and 8 stacked ensembles. XGBoost was skipped, because H2O's XGBoost is not available on Windows. By 5-fold cross-validation RMSE, the best model is StackedEnsemble_AllModels_3 (138.5 lb, test RMSE 134.6 lb, R² 0.761). The best single models are GBMs at about 138.9 lb, and the top of the leaderboard is mostly ensembles and GBMs. This seems to tell a few things about the athletes data. First, apart from one GBM grid model that failed to learn (258.4 lb, close to the label's standard deviation of 277 lb), all 48 other models land in a narrow range of about 138.5 to 152 lb. Even the plain linear GLM (146.4 lb) is only about 8 lb behind the best ensemble. A likely reason is that `total_lift` mostly rises and falls in a simple, close to additive way with a few inputs, so complex models have only a little extra pattern to find. Second, H2O's best test RMSE (134.6 lb) is almost the same as AutoGluon's (134.1 lb), even though the two tools trained different models in different ways. Two separate AutoML searches stopping at the same level suggests that about 134 lb is roughly the limit of what these 17 features can explain, not a limit of the algorithms. Third, the cross-validation RMSE is about 4 lb higher than the test RMSE for almost every model. A possible reason is that each cross-validation model is trained on only 80% of the training rows, so the cross-validation score seems to be a slightly pessimistic estimate for this dataset

### B.9.4 Top 5 features

In [15]:
# Stacked ensembles have no variable importance in H2O, so the best single model (a GBM)
# is used; importance = total squared-error reduction from splits on each feature
h2o_best_gbm = h2o_all.get_best_model(algorithm="gbm")
h2o_importance = h2o_best_gbm.varimp(use_pandas=True).set_index("variable")
H2O_TOP_FEATURES = h2o_importance.index[:TOP_N].tolist()

print("best single model:", h2o_best_gbm.model_id)
print("top 3 features (used in B.9.5):", H2O_TOP_FEATURES)
display(h2o_importance.head(5).round(3))

best single model: GBM_grid_1_AutoML_1_20261107_171216_model_19
top 3 features (used in B.9.5): ['gender', 'weight', 'bmi']


,relative_importance,scaled_importance,percentage
variable,,,
gender,2.623722e+09,1.000,0.397
weight,1.812183e+09,0.691,0.274
bmi,5.194921e+08,0.198,0.079
height,5.010964e+08,0.191,0.076
age,2.657202e+08,0.101,0.040


H2O's stacked ensembles do not report variable importance, so the importance below comes from the best single model, GBM_grid_1_AutoML_1_20261107_171216_model_19. H2O measures importance as the total drop in squared error from all tree splits on each feature.

| Rank | Feature | Share of total importance |
|---|---|---|
| 1 | gender | 39.7% |
| 2 | weight | 27.4% |
| 3 | bmi | 7.9% |
| 4 | height | 7.6% |
| 5 | age | 4.0% |

Like AutoGluon, H2O ranks `gender` first by a wide margin, and 4 of the 5 features (`gender`, `weight`, `bmi`, `age`) appear in both lists. The order is different, though: H2O puts `weight` second and includes `height`, while AutoGluon had `bmi` and `age` higher and `tenure` fifth. A likely reason is that the two tools measure importance in different ways. H2O counts how much each split improves the training fit, which tends to favor continuous features like `weight` and `height` that offer many possible split points. AutoGluon instead shuffles each feature and measures how much the validation score gets worse. So the top 3 features used in B.9.5 (`gender`, `weight`, `bmi`) are not the same as AutoGluon's (`gender`, `bmi`, `age`).

### B.9.5 Top 3 models per validation score

In [16]:
h2o_top, h2o_board_top = run_h2o(H2O_TOP_FEATURES, f"top{TOP_N}_features")

# Validation score = H2O's 5-fold cross-validation RMSE on train
for label, board in [("all features", h2o_board_all), (f"top {TOP_N} features", h2o_board_top)]:
    print(f"\nTop 3 models by validation RMSE, {label}:")
    with pd.option_context("display.max_colwidth", None):
        display(board.sort_values("rmse_val")[["model_id", "rmse_val", "rmse_test", "r2"]].head(3).round(3))


17:22:38.269: AutoML: XGBoost is not available; skipping it.



c:\Users\mpiir\AppData\Local\Programs\Python\Python312\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"



Top 3 models by validation RMSE, all features:


,model_id,rmse_val,rmse_test,r2
0,StackedEnsemble_AllModels_3_AutoML_1_20261107_171216,138.526,134.594,0.761
1,StackedEnsemble_BestOfFamily_4_AutoML_1_20261107_171216,138.567,134.575,0.761
2,StackedEnsemble_BestOfFamily_3_AutoML_1_20261107_171216,138.707,135.196,0.759



Top 3 models by validation RMSE, top 3 features:


,model_id,rmse_val,rmse_test,r2
0,StackedEnsemble_AllModels_3_AutoML_2_20261107_172238,173.348,171.923,0.610
1,StackedEnsemble_BestOfFamily_4_AutoML_2_20261107_172238,173.359,171.940,0.610
2,GBM_grid_1_AutoML_2_20261107_172238_model_10,173.423,171.970,0.609


| Rank | All 17 features | CV rmse_val | Top 3 features (gender, weight, bmi) | CV rmse_val |
|---|---|---|---|---|
| 1 | StackedEnsemble_AllModels_3 | 138.5 | StackedEnsemble_AllModels_3 | 173.3 |
| 2 | StackedEnsemble_BestOfFamily_4 | 138.6 | StackedEnsemble_BestOfFamily_4 | 173.4 |
| 3 | StackedEnsemble_BestOfFamily_3 | 138.7 | GBM_grid_1_model_10 | 173.4 |

With all features, all of the top 3 are stacked ensembles, and they are within 0.2 lb of each other. With only H2O's top 3 features, the two best models are still ensembles, but a single GBM is almost tied with them. Dropping to 3 features makes the models much worse: the best validation RMSE goes from about 138.5 lb to 173.3 lb, and the test R² drops from 0.761 to 0.610. This drop is bigger than AutoGluon's (whose top 3 features gave a test RMSE of about 162 lb, R² 0.654). A likely reason is that `weight` and `bmi` carry very similar information, since BMI is calculated from weight and height, so H2O's top 3 add less new information than AutoGluon's set, which includes `age` instead.